In [14]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week5_Spark_Assignment") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.2.0


In [5]:
df = spark.read.csv(
    r"C:\Sample - Superstore.csv",
    header=True,
    inferSchema=True
)

df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [15]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [16]:
print("Total Rows :", df.count())
print("Total Columns :", len(df.columns))

Total Rows : 9994
Total Columns : 21


# Q1. What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

## Answer

Traditional MapReduce has several limitations:

- Stores intermediate data on disk, resulting in slower execution.
- Performs poorly for iterative algorithms.
- Requires multiple MapReduce jobs for complex workflows.
- High disk I/O overhead.
- Limited support for real-time processing.

Apache Spark overcomes these limitations through in-memory computation, reducing execution time and supporting SQL, Streaming, Machine Learning, and Graph Processing within a single framework.

# Q2. Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

## Answer

Spark stores intermediate computation results in RAM instead of writing them to disk after every operation. Since machine learning algorithms repeatedly process the same data, keeping data in memory significantly reduces disk I/O and speeds up execution. This makes Spark much faster than traditional MapReduce.

# Q3. Write a code snippet to remove all duplicate rows from a DataFrame based on the columns user_id and transaction_date.

In [17]:
df_no_duplicates = df.dropDuplicates(["Order ID", "Order Date"])

df_no_duplicates.show()

+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+--------------+----------+-----------+-------+---------------+---------------+------------+--------------------+------------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|    Customer Name|    Segment|      Country|          City|     State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|       Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+--------------+----------+-----------+-------+---------------+---------------+------------+--------------------+------------+--------+--------+--------+
|  2718|CA-2014-100006|  9/7/2014| 9/13/2014|Standard Class|   DK-13375|      Dennis Kane|   Consumer|United States| New York City|  New York|      10024|   East|TEC-PH-10002075|     Technology

# Q4. Filter all transactions where the Region is "West", then group the data by Category and calculate the average Sales for each category.

In [11]:
from pyspark.sql.functions import avg

q4 = df.filter(df["Region"] == "West") \
       .groupBy("Category") \
       .agg(avg("Sales").alias("Average_Sales"))

q4.show()

+---------------+----------------+
|       Category|   Average_Sales|
+---------------+----------------+
|Office Supplies|116.422376910912|
|      Furniture|357.302324611033|
|     Technology|420.687532554257|
+---------------+----------------+



# Q5. Explain the difference between na.drop() and na.fill() with examples.

## Answer

- **na.drop()** removes rows containing null values.
- **na.fill()** replaces null values with a specified value without removing rows.

Example:

- `df.na.drop()` removes incomplete records.
- `df.na.fill(0)` replaces all null numeric values with 0.

# Q6. Count the total number of orders in each Region.

In [12]:
df.groupBy("Region").count().show()

+-------+-----+
| Region|count|
+-------+-----+
|  South| 1620|
|Central| 2323|
|   East| 2848|
|   West| 3203|
+-------+-----+



# Q7. Explain the difference between narrow transformations and wide transformations in Spark.

## Answer

A narrow transformation does not require data movement between partitions. Examples include `filter()` and `select()`.

A wide transformation requires data to be shuffled across partitions. Examples include `groupBy()`, `join()`, and `distinct()`.

Wide transformations are generally slower because of network communication.

# Q8. Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'.

In [ ]:
from pyspark.sql.functions import col

sample_data = [
    (21, "Premium"),
    (17, "Basic"),
    (25, "Premium"),
    (30, "Premium"),
    (35, "Premium")
]

sample_df = spark.createDataFrame(sample_data, ["age", "subscription"])

premium_users = sample_df.filter(
    (col("age").between(18, 30)) &
    (col("subscription") == "Premium")
)

premium_users.show()

# Q9. When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()?

## Answer

Handling null values before aggregation ensures accurate calculations. Null values may reduce the number of valid records considered or produce misleading results. Filling or removing null values improves data quality and ensures reliable statistical analysis.

# Q10. Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time.

In [ ]:
from pyspark.sql.types import TimestampType
from pyspark.sql.functions import col

sample_timestamp = spark.createDataFrame([
    ("2025-07-17 10:30:00",),
    ("2025-07-18 15:45:10",)
], ["raw_timestamp"])

sample_timestamp = sample_timestamp.withColumn(
    "event_time",
    col("raw_timestamp").cast(TimestampType())
).drop("raw_timestamp")

sample_timestamp.show(truncate=False)

# Q11. Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?

## Answer

A shuffle occurs when Spark redistributes data across partitions so that rows with the same key are grouped together. Since data moves between different partitions and worker nodes, it involves network communication and disk I/O. Therefore, operations like groupBy(), join(), and distinct() are called wide transformations and are generally slower than narrow transformations.

# Q12. Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string.

In [ ]:
from pyspark.sql.functions import col

clean_df = df.filter(
    col("email").isNotNull() &
    (col("username") != "")
)

clean_df.show()

# Q13. How do you use the .agg() function to calculate multiple statistics at once, such as the minimum, maximum, and mean of the price column?

In [ ]:
from pyspark.sql.functions import min, max, avg

df.agg(
    min("Sales").alias("Minimum Sales"),
    max("Sales").alias("Maximum Sales"),
    avg("Sales").alias("Average Sales")
).show()

# Q14. In the context of cleaning a dataset, what is the risk of using inferSchema=True when your source data contains messy or inconsistent date formats?

## Answer

When inferSchema=True is used on inconsistent date formats, Spark may infer incorrect data types or treat date values as strings. This can lead to parsing errors, incorrect sorting, failed date calculations, and inconsistent query results. For production systems, it is better to define the schema explicitly.

# Q15. Write a final processing pipeline that:
1. Filters out duplicates.
2. Fills null prices with 0.
3. Groups by store_id to calculate total revenue.

In [ ]:
from pyspark.sql.functions import sum

final_df = (
    df.dropDuplicates()
      .na.fill({"Sales":0})
      .groupBy("Customer ID")
      .agg(sum("Sales").alias("Total Sales"))
)

final_df.show()